# 01 · 데이터셋 감사 (NeuroLens CONDITION 학습 파이프라인)

이 노트북은 저장소의 `tools/colab/nlcolab` 패키지를 호출하는 **실행 화면**입니다. 코드는 노트북 안에 흩어져 있지 않고 저장소에 있습니다.

하는 일: ① 저장소 받기·의존성 설치 → ② 연구 관리자 계정으로 **인증된 내보내기**(기존 `dataset_list` / `dataset_detail` RPC, service_role 키 없음) → ③ 무결성·라벨·시간·좌표 검사 → ④ **불변 스냅샷**과 변경 탐지.

- 원시 자료는 Google Drive 의 `neurolens-training/raw` 에, 작업 결과는 `neurolens-training/work` 에 저장됩니다(Colab VM 은 꺼지면 사라집니다).
- 이 노트북은 모델을 바꾸지 않습니다. 운영 DB 에 쓰지 않습니다.
- Colab 은 24시간 켜 두는 서버가 아닙니다. 끊기면 다시 열어 같은 셀을 실행하면 체크포인트에서 이어집니다.

In [ ]:
#@title ① 저장소·의존성 (처음 한 번 · 재실행해도 안전)
import os, subprocess, sys
REPO = 'https://github.com/jeebs0627/neurolens.git'
if not os.path.isdir('/content/neurolens'):
    subprocess.check_call(['git', 'clone', '--depth', '50', REPO, '/content/neurolens'])
os.chdir('/content/neurolens')
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(), '← 학습 코드 commit (보고서에 기록됨)')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'tools/colab/requirements-colab.txt'])
if r.returncode != 0:
    print('기본 설치 실패 → CPU 전용 torch 로 재시도'); subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'scipy', 'scikit-learn', 'onnx', 'onnxruntime']); subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'])
print(subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip() or 'node 없음: apt-get install -y nodejs 필요', '← Node.js (기존 엔진 재현에 필요)')
sys.path.insert(0, '/content/neurolens/tools/colab')

In [ ]:
#@title ② Drive 연결 (raw/work 보존)
from google.colab import drive  # type: ignore
drive.mount('/content/drive')
RAW = '/content/drive/MyDrive/neurolens-training/raw'; WORK = '/content/drive/MyDrive/neurolens-training/work'
os.makedirs(RAW, exist_ok=True); os.makedirs(WORK, exist_ok=True); print(RAW, WORK)

In [ ]:
#@title ③ 인증된 내보내기 — 연구 관리자 로그인 (비밀번호는 화면에 남지 않고 저장되지 않음)
# 증분: 이미 받은 세션은 payload 해시가 같으면 annotations 만 갱신한다. 끝 페이지까지 읽은 근거는 raw/index.json 에 남는다.
import getpass
from nlcolab.export import sign_in, export_authenticated, DEFAULT_URL, DEFAULT_ANON
from nlcolab.safe_io import load_json
email = input('연구 관리자 이메일: ').strip(); token = sign_in(DEFAULT_URL, DEFAULT_ANON, email, getpass.getpass('비밀번호: '))
known = {s['id']: s.get('payload_sha256') for s in (load_json(f'{RAW}/index.json').get('sessions', []) if os.path.exists(f'{RAW}/index.json') else []) if s.get('payload_sha256')}
ev = export_authenticated(RAW, token, known_hashes=known)
del token
print({k: v for k, v in ev.items() if k != 'sessions'})

In [ ]:
#@title ③-b (선택) 기존 로컬 내보내기 파일 가져오기 — research_export/sessions/*.json 또는 dataset.html 내보내기 JSON 을 Drive 에 올린 뒤
from nlcolab.export import import_local
# import_local(RAW, ['/content/drive/MyDrive/neurolens-training/legacy_export/sessions'])

In [ ]:
#@title ④ 감사 → 인벤토리·적격성 보고서 (모든 기록이 사유와 함께 남는다)
from nlcolab.pipeline import Pipeline
pl = Pipeline(RAW, WORK)
inv = pl.audit()
print(open(f'{WORK}/inventory.md', encoding='utf-8').read())

In [ ]:
#@title ⑤ 불변 스냅샷 · 변경 탐지 (같은 자료·같은 버전이면 no_new_eligible_data)
snap, diff, trigger, reason = pl.snapshot(inv)
print('snapshot', snap['hash'][:16], snap['counts']); print('trigger', trigger, reason)
if diff.get('withdrawn'): print('⚠ 철회된 세션:', diff['withdrawn'], '— 이전 스냅샷·모델의 재학습/폐기 여부를 운영 절차로 결정하세요')

## 다음
`trigger` 가 True 이면 `02_train_gaze_residual.ipynb` 로 넘어갑니다. False 이면 새 적격 자료가 모일 때까지 학습하지 않습니다(중복 다운로드나 같은 프레임 추가만으로 재학습하지 않습니다).

인벤토리의 **제외 사유**(no-calibration-frames, target-windows-inferred-backfill, legacy-vertical-feature-not-reconstructible, reference-sync-unverified …)는 수집 쪽에서 무엇을 보완해야 하는지 보여 줍니다. 요약만 남은 세션은 프레임 자료로 복원하지 않습니다.